In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
from Informatics import softmax, markov_process
data = pd.read_csv("Linker dataset.csv")
tpm,amino_acids = markov_process(data)

In [10]:
olc = amino_acids
lst = list(np.random.randint(len(amino_acids),size = 10))
seq = ''
for i in lst:
    seq+=amino_acids[i]
mut = ['ins','del','sub','ret']
mutp = [1/5,1/5,1/2,1.5/5]

In [11]:
print('initial sequence : ',seq,'\n','_'*100)


initial sequence :  AQFHCPTSTP 
 ____________________________________________________________________________________________________


In [ ]:
# Algorithemic parameters
mutp = softmax(mutp)      # Probability For Different mutations
num_gen = 20              
top_k = 5
num_daughter = 1000       # Number Daughter linker samples per linker
def evolution(seq):
    out = ''
    for i in range(len(seq)):
        choice = np.random.choice(mut, p=mutp)              # Choosing the type of mutation
        if choice == 'ret':                                 # Retention - Keep the same
            out += seq[i]
        elif choice == 'del':                               # Deletion - Remove the Amino Acid
            pass
        elif choice == 'sub':                               # Substitution
            if seq[i-1]:                                    
                out += np.random.choice(olc, p=tpm[seq[i-1]])   # Choose based on Transition probability Matrix from the previous Amino acid
            else:
                out += seq[i]                                   # if first Amino acid then keep the same
        elif choice == 'ins':
            out = out + i + np.random.choice(olc, p=tpm[seq[i]]) # Choose based on Transition probability Matrix from the previous Amino acid
        else: raise ValueError('Mutation Choie error')
    return out
            
def weighted_sum(arr):
    return arr[0] +  arr[1] + arr[2] +  arr[3] +  arr[4]

In [ ]:
def pareto_dominates(parent, daughter):                     # Returns True if parent dominates daughter in every function aspect.
    if len(parent) != len(daughter):                        # All elements must be numeric and of the same length.
        raise ValueError("Vectors must have the same length")
    else:
        for i in range(len(parent)):
            if parent[i]>daughter[i]:
                return False
    return True

In [ ]:
def reward_fn(seq):
    return None

In [ ]:
k = num_gen
generation = {}
def GA(seq_l):          # Establishing the Genetic Algorithm
    global k
    global out_l2
    if k > 0:
        out_l1 = []     # Stores All daughter linkers
        out_l2 = []     # Stored all the filtered linkers
        
        k -= 1
        for i in seq_l:     
            for j in range(num_daughter):
                out_l1.append(evolution(i))
        for i in out_l1:
            if pareto_dominates(reward_fn(seq_l[0]),reward_fn(i)):      # If pareto retuns true, the linker gets appended.
                out_l2.append(i)
        out_l2.sort(key = lambda x:weighted_sum(reward_fn(x)))          # Sorting according to some weighted sum, Can use some other paramtere also
        generation[f'Generation Number {num_gen - k}'] = out_l2[:top_k]
        return GA(out_l2[:top_k])
    else:
        return out_l2
    
